# Energy Generation & Meter Reading Dashboard

In [ ]:

import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
from pathlib import Path
from datetime import date
import math

# Jupyter Notebook does not define __file__.
# The database is created in the notebook current working folder.
DB_PATH = Path.cwd() / "energy_dashboard.db"


def get_connection():
    connection = sqlite3.connect(DB_PATH)
    cursor = connection.cursor()
    cursor.execute("PRAGMA foreign_keys = ON")
    return connection


def create_tables():
    connection = get_connection()
    cursor = connection.cursor()
    try:
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS sites (
                site_id INTEGER PRIMARY KEY AUTOINCREMENT,
                site_name TEXT NOT NULL UNIQUE,
                contact_email TEXT NOT NULL
            )
        """)

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS generation_reports (
                report_id INTEGER PRIMARY KEY AUTOINCREMENT,
                site_id INTEGER NOT NULL,
                energy_source TEXT NOT NULL,
                report_date TEXT NOT NULL,
                FOREIGN KEY (site_id) REFERENCES sites(site_id)
                    ON DELETE CASCADE
            )
        """)

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS energy_readings (
                reading_id INTEGER PRIMARY KEY AUTOINCREMENT,
                report_id INTEGER NOT NULL,
                metric_name TEXT NOT NULL,
                metric_value REAL NOT NULL CHECK (metric_value >= 0),
                FOREIGN KEY (report_id) REFERENCES generation_reports(report_id)
                    ON DELETE CASCADE
            )
        """)
        connection.commit()
    except sqlite3.Error:
        connection.rollback()
        raise
    finally:
        connection.close()


def add_energy_record(site_name, contact_email, energy_source,
                      metric_name, metric_value):
    site_name = site_name.strip()
    contact_email = contact_email.strip()
    energy_source = energy_source.strip()
    metric_name = metric_name.strip()

    if not site_name or not contact_email or not energy_source or not metric_name:
        raise ValueError("Please fill in all required fields.")

    try:
        value = float(metric_value)
    except (TypeError, ValueError):
        raise ValueError("Metric Value must be numeric.")

    if not math.isfinite(value) or value < 0:
        raise ValueError("Metric Value must be a valid non-negative number.")

    connection = get_connection()
    cursor = connection.cursor()
    try:
        cursor.execute(
            "SELECT site_id FROM sites WHERE site_name = ?",
            (site_name,)
        )
        site = cursor.fetchone()

        if site:
            # Reuse the existing Site ID without changing its saved email.
            site_id = site[0]
        else:
            cursor.execute(
                "INSERT INTO sites (site_name, contact_email) VALUES (?, ?)",
                (site_name, contact_email)
            )
            site_id = cursor.lastrowid

        cursor.execute(
            """INSERT INTO generation_reports
               (site_id, energy_source, report_date)
               VALUES (?, ?, ?)""",
            (site_id, energy_source, date.today().isoformat())
        )
        report_id = cursor.lastrowid

        cursor.execute(
            """INSERT INTO energy_readings
               (report_id, metric_name, metric_value)
               VALUES (?, ?, ?)""",
            (report_id, metric_name, value)
        )
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()


def get_joined_records():
    connection = get_connection()
    cursor = connection.cursor()
    try:
        cursor.execute("""
            SELECT generation_reports.report_id,
                   sites.site_name,
                   generation_reports.energy_source,
                   generation_reports.report_date,
                   energy_readings.metric_name,
                   energy_readings.metric_value
            FROM generation_reports
            INNER JOIN sites
                ON generation_reports.site_id = sites.site_id
            INNER JOIN energy_readings
                ON generation_reports.report_id = energy_readings.report_id
            ORDER BY generation_reports.report_id
        """)
        return cursor.fetchall()
    finally:
        connection.close()


def get_table_records(table_name):
    table_names = {
        "sites": "sites",
        "generation_reports": "generation_reports",
        "energy_readings": "energy_readings"
    }

    if table_name not in table_names:
        raise ValueError("Invalid table name.")

    connection = get_connection()
    cursor = connection.cursor()
    try:
        cursor.execute(
            "SELECT * FROM " + table_names[table_name] + " ORDER BY 1"
        )
        return cursor.fetchall()
    finally:
        connection.close()


def delete_record(record_type, record_id):
    delete_options = {
        "Site": ("sites", "site_id"),
        "Generation Report": ("generation_reports", "report_id"),
        "Energy Reading": ("energy_readings", "reading_id")
    }

    if record_type not in delete_options:
        raise ValueError("Please select a valid record type.")

    table_name, id_column = delete_options[record_type]
    connection = get_connection()
    cursor = connection.cursor()
    try:
        cursor.execute(
            "DELETE FROM " + table_name + " WHERE " + id_column + " = ?",
            (record_id,)
        )
        deleted_count = cursor.rowcount
        connection.commit()
        return deleted_count
    except sqlite3.Error:
        connection.rollback()
        raise
    finally:
        connection.close()


def add_demo_record_if_database_is_empty():
    """Add one example row on first run so the dashboard is not blank."""
    connection = get_connection()
    cursor = connection.cursor()
    try:
        cursor.execute("SELECT COUNT(*) FROM sites")
        site_count = cursor.fetchone()[0]
        cursor.execute("SELECT COUNT(*) FROM generation_reports")
        report_count = cursor.fetchone()[0]
        cursor.execute("SELECT COUNT(*) FROM energy_readings")
        reading_count = cursor.fetchone()[0]

        if site_count == 0 and report_count == 0 and reading_count == 0:
            cursor.execute(
                "INSERT INTO sites (site_name, contact_email) VALUES (?, ?)",
                ("Demo Solar Site", "demo@example.com")
            )
            site_id = cursor.lastrowid

            cursor.execute(
                """INSERT INTO generation_reports
                   (site_id, energy_source, report_date)
                   VALUES (?, ?, ?)""",
                (site_id, "Solar", date.today().isoformat())
            )
            report_id = cursor.lastrowid

            cursor.execute(
                """INSERT INTO energy_readings
                   (report_id, metric_name, metric_value)
                   VALUES (?, ?, ?)""",
                (report_id, "Energy Generated (kWh)", 250.0)
            )
            connection.commit()
    except sqlite3.Error:
        connection.rollback()
        raise
    finally:
        connection.close()


def launch_dashboard():
    root = tk.Tk()
    root.title("Energy Generation & Meter Reading Dashboard")
    root.geometry("1250x780")
    root.minsize(1000, 650)

    style = ttk.Style(root)
    try:
        style.theme_use("clam")
    except tk.TclError:
        pass

    main_frame = ttk.Frame(root, padding=12)
    main_frame.pack(fill="both", expand=True)

    heading = ttk.Label(
        main_frame,
        text="ENERGY GENERATION & METER READING DASHBOARD",
        font=("Arial", 17, "bold")
    )
    heading.pack(pady=(0, 12))

    # -------------------- INSERT PANEL --------------------
    insert_frame = ttk.LabelFrame(main_frame, text="Insert Panel", padding=10)
    insert_frame.pack(fill="x", pady=5)

    site_name_var = tk.StringVar()
    contact_email_var = tk.StringVar()
    energy_source_var = tk.StringVar()
    metric_name_var = tk.StringVar()
    metric_value_var = tk.StringVar()

    input_fields = [
        ("Site Name", site_name_var),
        ("Contact Email", contact_email_var),
        ("Energy Source", energy_source_var),
        ("Metric Description", metric_name_var),
        ("Metric Value", metric_value_var)
    ]

    for column, (label_text, variable) in enumerate(input_fields):
        ttk.Label(insert_frame, text=label_text).grid(
            row=0, column=column, padx=5, sticky="w"
        )
        ttk.Entry(
            insert_frame, textvariable=variable, width=22
        ).grid(row=1, column=column, padx=5, pady=5, sticky="ew")
        insert_frame.columnconfigure(column, weight=1)

    # -------------------- DELETE PANEL --------------------
    delete_frame = ttk.LabelFrame(main_frame, text="Delete Panel", padding=10)
    delete_frame.pack(fill="x", pady=6)

    delete_type_var = tk.StringVar(value="Site")
    delete_id_var = tk.StringVar()

    ttk.Label(delete_frame, text="Record Type").pack(side="left", padx=(0, 5))

    type_dropdown = ttk.Combobox(
        delete_frame,
        textvariable=delete_type_var,
        values=["Site", "Generation Report", "Energy Reading"],
        state="readonly",
        width=22
    )
    type_dropdown.pack(side="left", padx=5)
    type_dropdown.current(0)

    ttk.Label(delete_frame, text="Record ID").pack(side="left", padx=(12, 5))

    id_dropdown = ttk.Combobox(
        delete_frame,
        textvariable=delete_id_var,
        state="readonly",
        width=18
    )
    id_dropdown.pack(side="left", padx=5)

    # -------------------- DISPLAY TABS --------------------
    tabs = ttk.Notebook(main_frame)
    tabs.pack(fill="both", expand=True, pady=8)

    table_details = {
        "Combined Report": (
            "joined",
            ["Report ID", "Site Name", "Energy Source",
             "Report Date", "Metric Name", "Metric Value"]
        ),
        "Sites": (
            "sites",
            ["Site ID", "Site Name", "Contact Email"]
        ),
        "Generation Reports": (
            "generation_reports",
            ["Report ID", "Site ID", "Energy Source", "Report Date"]
        ),
        "Energy Readings": (
            "energy_readings",
            ["Reading ID", "Report ID", "Metric Name", "Metric Value"]
        )
    }

    treeviews = {}

    for tab_title, (table_name, headings) in table_details.items():
        tab_frame = ttk.Frame(tabs, padding=6)
        tabs.add(tab_frame, text=tab_title)

        table_frame = ttk.Frame(tab_frame)
        table_frame.pack(fill="both", expand=True)

        tree = ttk.Treeview(
            table_frame, columns=headings, show="headings"
        )
        vertical_scroll = ttk.Scrollbar(
            table_frame, orient="vertical", command=tree.yview
        )
        horizontal_scroll = ttk.Scrollbar(
            table_frame, orient="horizontal", command=tree.xview
        )

        tree.configure(
            yscrollcommand=vertical_scroll.set,
            xscrollcommand=horizontal_scroll.set
        )

        tree.grid(row=0, column=0, sticky="nsew")
        vertical_scroll.grid(row=0, column=1, sticky="ns")
        horizontal_scroll.grid(row=1, column=0, sticky="ew")

        table_frame.rowconfigure(0, weight=1)
        table_frame.columnconfigure(0, weight=1)

        for heading_text in headings:
            tree.heading(heading_text, text=heading_text)
            tree.column(
                heading_text, width=160, minwidth=100, anchor="center"
            )

        treeviews[table_name] = tree

    def refresh_delete_ids():
        selected_type = delete_type_var.get()

        delete_table_details = {
            "Site": ("sites", "site_id"),
            "Generation Report": ("generation_reports", "report_id"),
            "Energy Reading": ("energy_readings", "reading_id")
        }

        table_name, id_column = delete_table_details[selected_type]

        connection = get_connection()
        cursor = connection.cursor()
        try:
            cursor.execute(
                "SELECT " + id_column + " FROM " + table_name +
                " ORDER BY " + id_column
            )
            rows = cursor.fetchall()
        finally:
            connection.close()

        available_ids = [str(row[0]) for row in rows]
        id_dropdown["values"] = available_ids

        if delete_id_var.get() not in available_ids:
            if available_ids:
                delete_id_var.set(available_ids[0])
            else:
                delete_id_var.set("")

    def refresh_tables():
        for table_name, tree in treeviews.items():
            for item in tree.get_children():
                tree.delete(item)

            if table_name == "joined":
                rows = get_joined_records()
            else:
                rows = get_table_records(table_name)

            for row in rows:
                tree.insert("", "end", values=row)

        refresh_delete_ids()

    def add_record():
        try:
            add_energy_record(
                site_name_var.get(),
                contact_email_var.get(),
                energy_source_var.get(),
                metric_name_var.get(),
                metric_value_var.get()
            )

            site_name_var.set("")
            contact_email_var.set("")
            energy_source_var.set("")
            metric_name_var.set("")
            metric_value_var.set("")

            refresh_tables()
            messagebox.showinfo(
                "Success", "Record added successfully.", parent=root
            )

        except ValueError as error:
            messagebox.showerror(
                "Input Error", str(error), parent=root
            )
        except sqlite3.Error as error:
            messagebox.showerror(
                "Database Error", str(error), parent=root
            )

    def delete_selected():
        selected_type = delete_type_var.get()
        selected_id = delete_id_var.get()

        if not selected_id:
            messagebox.showwarning(
                "Select Record",
                "There is no record ID available for this record type.",
                parent=root
            )
            return

        confirm = messagebox.askyesno(
            "Confirm Delete",
            "Delete " + selected_type + " ID " + selected_id +
            "?\nDeleting a parent may also delete its related records.",
            parent=root
        )

        if not confirm:
            return

        try:
            deleted_count = delete_record(selected_type, int(selected_id))
            site_name_var.set("")
            contact_email_var.set("")
            energy_source_var.set("")
            metric_name_var.set("")
            metric_value_var.set("")
            refresh_tables()

            if deleted_count > 0:
                messagebox.showinfo(
                    "Deleted", "Selected record deleted successfully.",
                    parent=root
                )
            else:
                messagebox.showwarning(
                    "Not Found", "The selected record no longer exists.",
                    parent=root
                )

        except (ValueError, sqlite3.Error) as error:
            messagebox.showerror(
                "Delete Error", str(error), parent=root
            )

    ttk.Button(
        insert_frame, text="Add Record", command=add_record
    ).grid(row=2, column=0, columnspan=5, pady=(8, 0))

    ttk.Button(
        delete_frame, text="Delete Selected", command=delete_selected
    ).pack(side="left", padx=12)

    type_dropdown.bind(
        "<<ComboboxSelected>>",
        lambda event: refresh_delete_ids()
    )

    refresh_tables()
    print("Dashboard opened. Use the window to add, view and delete records.")
    root.mainloop()


if __name__ == "__main__":
    create_tables()
    add_demo_record_if_database_is_empty()
    print("Database ready:", DB_PATH)
    print("Combined report rows:", len(get_joined_records()))
    print("Sites:", len(get_table_records("sites")))
    print("Generation reports:", len(get_table_records("generation_reports")))
    print("Energy readings:", len(get_table_records("energy_readings")))
    launch_dashboard()


Database ready: C:\Users\Admin\dbms\energy_dashboard.db
Combined report rows: 1
Sites: 1
Generation reports: 1
Energy readings: 1
Dashboard opened. Use the window to add, view and delete records.
